# Hospital Readmission Risk Predictor
## Baseline Models Notebook

**Author:** Angeline Setiawan
**Team:** Epsilon
**GitHub:** https://github.com/AngelineSetiawan/AngelineSetiawan-hospital-readmission-risk-predictor

This notebook builds the model-ready table established in the EDA Report
(July 21, 2026) and the Preprocessing & Feature Engineering Report (July 28,
2026), then fits and evaluates the two baseline models specified in that
report's modeling plan: a dummy majority-class classifier and a regularized,
class-weighted logistic regression.

Run all cells top to bottom. Each section prints the numbers needed for the
Baseline Models Report; copy the printed output back for write-up.


## 0. Setup

In [2]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score, f1_score,
    brier_score_loss, confusion_matrix, roc_curve
)
from sklearn.calibration import calibration_curve

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
plt.rcParams.update({"font.size": 11, "figure.dpi": 130, "savefig.bbox": "tight"})

# Update these two paths to wherever your raw files live locally / in Colab.
RAW_PATH = "diabetic_data.csv"
IDS_PATH = "IDS_mapping.csv"


## 1. Preprocessing Pipeline

This section reproduces, in order, every cleaning and feature-engineering
decision documented in the EDA and Preprocessing reports, so the baseline
model below is trained on the same model-ready table those reports describe,
not a re-derived approximation of it. Each step states what is done and why,
consistent with the reasoning already on record in those two reports.


In [3]:
# ============================================================
# File paths — adjust here if your repo structure differs
# ============================================================
RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"

RAW_ENCOUNTERS_PATH = f"{RAW_DATA_DIR}/diabetic_data.csv"
IDS_MAPPING_PATH = f"{RAW_DATA_DIR}/IDS_mapping.csv"

TRAIN_OUT_PATH = f"{PROCESSED_DATA_DIR}/train.csv"
TEST_OUT_PATH = f"{PROCESSED_DATA_DIR}/test.csv"

In [4]:
# ----------------------------------------------------------------------
# 1.1 Load raw data
# ----------------------------------------------------------------------
df = pd.read_csv(RAW_ENCOUNTERS_PATH)
n_raw = len(df)
print(f"Raw encounters loaded: {n_raw:,}")

df.head()


Raw encounters loaded: 101,766


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [ ]:
# ----------------------------------------------------------------------
# 1.2 Standardize "?" placeholders to true nulls.
# Why: this is a prerequisite for accurate missingness counting and for
# every downstream missing-data decision to behave predictably (EDA
# Report, Section 2.2).
# ----------------------------------------------------------------------
df = df.replace("?", np.nan)


In [ ]:
# ----------------------------------------------------------------------
# 1.3 Parse IDS_mapping.csv (three stacked lookup tables separated by
# blank rows) into three id -> label dictionaries, then join each onto
# the encounter table.
# Why: the raw numeric codes are not interpretable to a non-technical
# stakeholder; human-readable labels make the categorical summaries in
# this report legible (EDA Report, Section 2.1).
# ----------------------------------------------------------------------
ids_raw = pd.read_csv(IDS_PATH, header=None)

# Each sub-table begins with a row whose second column is the literal
# string "description" -- that marks a new block's header row.
header_rows = ids_raw.index[ids_raw[1] == "description"].tolist()
header_rows.append(len(ids_raw))

lookup = {}
for i in range(len(header_rows) - 1):
    start, end = header_rows[i], header_rows[i + 1]
    block = ids_raw.iloc[start:end].copy()
    id_col_name = block.iloc[0, 0]          # e.g. "admission_type_id"
    block = block.iloc[1:]                   # drop the header row
    block = block.dropna(how="all")
    block[0] = pd.to_numeric(block[0], errors="coerce")
    block = block.dropna(subset=[0])
    lookup[id_col_name] = dict(zip(block[0].astype(int), block[1]))

for id_col, label_col in [
    ("admission_type_id", "admission_type_id_label"),
    ("discharge_disposition_id", "discharge_disposition_id_label"),
    ("admission_source_id", "admission_source_id_label"),
]:
    df[label_col] = df[id_col].map(lookup[id_col])
    # The source reference table has inconsistent leading/trailing
    # whitespace on some label strings (e.g. " Emergency Room"). Strip it
    # so identical categories aren't fragmented into near-duplicates.
    df[label_col] = df[label_col].str.strip()

print("Label columns created:")
for c in ["admission_type_id_label", "discharge_disposition_id_label", "admission_source_id_label"]:
    print(f"  {c}: {df[c].nunique()} categories")


In [ ]:
# ----------------------------------------------------------------------
# 1.4 Consolidate hidden-unknown placeholder strings into one "Unknown"
# category per label field.
# Why: leaving "NULL", "Not Available", "Not Mapped", and "Unknown/Invalid"
# as separate categories fragments an already sparse signal across
# near-duplicate labels for no analytical benefit (Preprocessing Report,
# "Unknown-category consolidation").
# ----------------------------------------------------------------------
unknown_strings = {"NULL", "Not Available", "Not Mapped", "Unknown/Invalid"}
for label_col in [
    "admission_type_id_label",
    "discharge_disposition_id_label",
    "admission_source_id_label",
]:
    df[label_col] = df[label_col].apply(
        lambda x: "Unknown" if (pd.isna(x) or x in unknown_strings) else x
    )


In [ ]:
# ----------------------------------------------------------------------
# 1.5 Derive the binary target and drop the native 3-class label.
# Why: readmitted_30d aligns the modeling target directly with the
# outcome CMS measures under HRRP. The native label is then removed so
# it cannot leak into the model as a restated version of the target
# it is meant to predict (Preprocessing Report, "Target derivation").
# ----------------------------------------------------------------------
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)
df = df.drop(columns=["readmitted"])

print("Target prevalence (all encounters):", round(df["readmitted_30d"].mean() * 100, 2), "%")


In [ ]:
# ----------------------------------------------------------------------
# 1.6 Drop weight.
# Why: 96.86% missing with no recoverable pattern; no imputation strategy
# at this sample size would be reliable (EDA Report, Section 3.3;
# Preprocessing Report, "Weight removal").
# ----------------------------------------------------------------------
df = df.drop(columns=["weight"])


In [ ]:
# ----------------------------------------------------------------------
# 1.7 Testing indicators for the two lab-result fields.
# Why: missingness in max_glu_serum / A1Cresult reflects a physician's
# decision not to order the test, not a data gap. Encoding "tested vs.
# not tested" preserves that clinical signal; imputing a plausible lab
# value would fabricate a result that was never recorded (Preprocessing
# Report, "Testing indicators" and "Imputation Alternatives Considered
# and Rejected").
# ----------------------------------------------------------------------
df["max_glu_serum_tested"] = np.where(df["max_glu_serum"].isna(), "Not Tested", "Tested")
df["A1Cresult_tested"] = np.where(df["A1Cresult"].isna(), "Not Tested", "Tested")


In [ ]:
# ----------------------------------------------------------------------
# 1.8 Explicit missing category for demographic / administrative fields.
# Why: for race, mode-imputing the majority category would introduce a
# systematic, direction-biased distortion into a protected attribute.
# For medical_specialty / payer_code, mode imputation would overweight
# the majority category and could mask a genuine administrative pattern.
# An explicit category preserves every row without fabricating a label
# (Preprocessing Report, Table 2b).
# ----------------------------------------------------------------------
for col in ["race", "medical_specialty", "payer_code"]:
    df[col] = df[col].fillna("Missing")


In [ ]:
# ----------------------------------------------------------------------
# 1.9 Mortality exclusion.
# Why: discharge disposition codes 11, 19, 20, and 21 are in-hospital
# deaths. Every one of these rows is coded "not readmitted" by
# construction, since a deceased patient cannot be readmitted -- these
# are guaranteed negative-class examples for a reason that has nothing
# to do with clinical risk. Consistent with Strack et al. (2014) and the
# Preprocessing Report ("Mortality exclusion"), these rows are removed
# rather than left in as ordinary negative examples.
# ----------------------------------------------------------------------
mortality_codes = [11, 19, 20, 21]
n_before_mortality = len(df)
df = df[~df["discharge_disposition_id"].isin(mortality_codes)].copy()
n_after_mortality = len(df)

print(f"Removed {n_before_mortality - n_after_mortality:,} in-hospital-death encounters")
print(f"Remaining: {n_after_mortality:,}")


In [ ]:
# ----------------------------------------------------------------------
# 1.10 Diagnosis code grouping (diag_1 only; diag_2 / diag_3 are grouped
# with the same function but held out of this baseline feature set --
# see the note in Section 2 below on baseline scope).
# Why: diag_1 contains hundreds of distinct ICD-9 codes. One-hot encoding
# them directly would create a feature space large relative to the
# number of available rows and would fragment clinically related
# conditions into hundreds of near-empty columns. Grouping into 9
# standard ICD-9 chapters keeps the signal usable and lets a clinical
# reviewer reason about "a circulatory diagnosis" rather than code
# 414.01 (Preprocessing Report, Section 2.3).
# ----------------------------------------------------------------------
def group_icd9(code_str):
    if pd.isna(code_str):
        return "Missing"
    code_str = str(code_str)
    if code_str.startswith("250"):
        return "Diabetes"
    if code_str.startswith(("V", "E")):
        return "Other/Unclassified"
    try:
        val = float(code_str)
    except ValueError:
        return "Other/Unclassified"
    if (390 <= val <= 459) or val == 785:
        return "Circulatory"
    if (460 <= val <= 519) or val == 786:
        return "Respiratory"
    if (520 <= val <= 579) or val == 787:
        return "Digestive"
    if 800 <= val <= 999:
        return "Injury"
    if 710 <= val <= 739:
        return "Musculoskeletal"
    if (580 <= val <= 629) or val == 788:
        return "Genitourinary"
    if 140 <= val <= 239:
        return "Neoplasms"
    return "Other/Unclassified"

df["diag_1_group"] = df["diag_1"].apply(group_icd9)
print(df["diag_1_group"].value_counts())


In [ ]:
# ----------------------------------------------------------------------
# 1.11 Chronology proxy (encounter_order).
# Why: the public release has no exact calendar date. encounter_id is
# used as an ordinal proxy for the order encounters were extracted from
# the source system, which is what makes a genuinely time-ordered
# train/test split possible (Proposal, "Accounting for time structure").
# ----------------------------------------------------------------------
df = df.sort_values("encounter_id").reset_index(drop=True)
df["encounter_order"] = np.arange(len(df))


In [ ]:
# ----------------------------------------------------------------------
# 1.12 Patient-level snapshot: keep each patient's first eligible
# encounter (earliest by encounter_order); drop all later encounters for
# the same patient.
# Why: 23.4% of patients have more than one encounter. Treating each
# encounter as independent would let a single patient's history leak
# across train/test and would distort feature-importance estimates
# toward frequently readmitted patients -- nearly every standard
# classifier and evaluation metric assumes independent rows
# (Preprocessing Report, Section 2.1 and 2.3).
# ----------------------------------------------------------------------
n_before_snapshot = len(df)
snapshot = df.sort_values("encounter_order").groupby("patient_nbr", as_index=False).first()
snapshot = snapshot.sort_values("encounter_order").reset_index(drop=True)
n_after_snapshot = len(snapshot)

print(f"Encounters before snapshot: {n_before_snapshot:,}")
print(f"Unique patients (snapshot rows): {n_after_snapshot:,}")
print(f"Snapshot readmission rate: {round(snapshot['readmitted_30d'].mean() * 100, 2)}%")


In [ ]:
# ----------------------------------------------------------------------
# 1.13 Time-aware, patient-grouped 75:25 split (chronological, not
# random).
# Why: the eventual model will always be scored against patients who
# come after the ones it was trained on. A random split would leak
# future information into training. Splitting on the same ordinal proxy
# used for the snapshot keeps the split honest to that constraint
# (Preprocessing Report, Section 3.4).
# ----------------------------------------------------------------------
cut = int(len(snapshot) * 0.75)
train = snapshot.iloc[:cut].copy()
test = snapshot.iloc[cut:].copy()

assert set(train["patient_nbr"]).isdisjoint(set(test["patient_nbr"])), "Patient overlap detected!"

print(f"Train rows: {len(train):,}  | readmit rate: {round(train['readmitted_30d'].mean() * 100, 2)}%")
print(f"Test rows:  {len(test):,}  | readmit rate: {round(test['readmitted_30d'].mean() * 100, 2)}%")
print("Patient overlap check passed: 0 shared patients between train and test.")


In [ ]:
# Save the split tables so they can be reloaded without re-running the
# pipeline (useful if this notebook is split across multiple sessions).
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)
snapshot.to_csv("snapshot_full.csv", index=False)


## 2. Baseline Feature Set

The baseline model uses the "core" predictor set already established across
the EDA and Preprocessing reports: the eight utilization/complexity variables
central to the project's hypothesis, demographics, admission/discharge
context, diagnosis category, lab-testing indicators, and treatment-activity
flags.

**Scope note:** two fields the Preprocessing Report flagged for *supervised*
feature engineering -- target-encoded `medical_specialty` and diag_2 /
diag_3 grouping -- are deliberately held out of this baseline. That work is
scheduled for the modeling phase, not the baseline week, so this model
reflects only cleaning and the feature engineering already completed and
documented, not feature engineering that is still planned.


In [ ]:
TARGET = "readmitted_30d"

CONTINUOUS = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses",
]

CATEGORICAL = [
    "race", "gender", "admission_type_id_label",
    "discharge_disposition_id_label", "admission_source_id_label",
    "max_glu_serum_tested", "A1Cresult_tested", "change", "diabetesMed",
    "insulin", "metformin", "diag_1_group",
]

# Age is naturally ordinal (10-year brackets). Encoding it as an ordinal
# integer, rather than one-hot, preserves the fact that order carries real
# information -- the EDA Report (Figure 4) showed a broadly monotonic
# relationship between age bracket and readmission risk.
AGE_ORDER = ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
             "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]
age_map = {v: i for i, v in enumerate(AGE_ORDER)}


def build_design_matrix(data, ref_columns=None):
    """One-hot encode categoricals and assemble the full design matrix.
    If ref_columns is given (from the training fit), align the new matrix
    to those exact columns so categories unseen in this split don't break
    the model (e.g. a category present in test but not train)."""
    X = data.copy()
    X["age_ord"] = X["age"].map(age_map)
    cat_dummies = pd.get_dummies(X[CATEGORICAL], drop_first=True)
    X_full = pd.concat([X[CONTINUOUS], X[["age_ord"]], cat_dummies], axis=1)
    X_full = X_full.astype(float)
    if ref_columns is not None:
        X_full = X_full.reindex(columns=ref_columns, fill_value=0.0)
    return X_full


X_train_raw = build_design_matrix(train)
feature_columns = X_train_raw.columns.tolist()
X_test_raw = build_design_matrix(test, ref_columns=feature_columns)

y_train = train[TARGET].values
y_test = test[TARGET].values

# Scale continuous features only; the scaler is fit on training data ONLY
# to avoid any leakage from the holdout set into the scaling parameters.
scaler = StandardScaler()
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
X_train[CONTINUOUS + ["age_ord"]] = scaler.fit_transform(X_train_raw[CONTINUOUS + ["age_ord"]])
X_test[CONTINUOUS + ["age_ord"]] = scaler.transform(X_test_raw[CONTINUOUS + ["age_ord"]])

print("Design matrix shape (train):", X_train.shape)
print("Design matrix shape (test): ", X_test.shape)
print("Total features (incl. one-hot dummies):", len(feature_columns))


## 3. Dummy Majority-Class Baseline

This sets the minimum bar: any model with real predictive value must clear
this baseline's metrics on AUC-ROC and recall for the readmitted class, per
the evaluation criteria set out in the original proposal.


In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
dummy_pred_test = dummy.predict(X_test)

dummy_results = {
    "test_accuracy": float((dummy_pred_test == y_test).mean()),
    "test_recall": float(recall_score(y_test, dummy_pred_test, zero_division=0)),
    "test_precision": float(precision_score(y_test, dummy_pred_test, zero_division=0)),
    "test_auc": 0.5,  # a constant-probability classifier has no discrimination
}

print("Dummy majority-class baseline (holdout test set):")
for k, v in dummy_results.items():
    print(f"  {k}: {v}")


## 4. Logistic Regression Baseline

**Why logistic regression as the baseline:** it is the simplest reasonable
and most interpretable classifier for this problem, consistent with the
Proposal's modeling plan. It gives directly interpretable coefficients
(odds ratios) that a care-management stakeholder can reason about, and it
sets an honest floor that a more complex model (random forest, XGBoost) must
clear to justify its added complexity.

**Class imbalance handling:** `class_weight="balanced"` re-weights the loss
function so the minority (readmitted) class is not ignored, consistent with
the proposal's stated preference for class weighting as the primary
imbalance strategy, with resampling (SMOTE) evaluated only as a comparison
in a later phase.

**Cross-validation:** 5-fold **stratified** CV, run only on the training
partition (52,829 rows). Stratification keeps the ~9.6% positive-class rate
consistent across folds, which matters given how imbalanced the target is.
Cross-validation here is the primary control for overfitting at the baseline
stage; the holdout test set is touched only once, at the end, for a final
check.


In [ ]:
logreg = LogisticRegression(
    C=1.0, solver="lbfgs", max_iter=2000,
    class_weight="balanced", random_state=42,
)
logreg.fit(X_train, y_train)

# 5-fold stratified cross-validation on the TRAINING partition only.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    logreg, X_train, y_train, cv=cv,
    scoring=["roc_auc", "recall", "precision", "f1"],
    return_train_score=True,
)

cv_summary = {
    "cv_val_auc_mean": float(np.mean(cv_results["test_roc_auc"])),
    "cv_val_auc_std": float(np.std(cv_results["test_roc_auc"])),
    "cv_train_auc_mean": float(np.mean(cv_results["train_roc_auc"])),
    "cv_val_recall_mean": float(np.mean(cv_results["test_recall"])),
    "cv_val_recall_std": float(np.std(cv_results["test_recall"])),
    "cv_val_precision_mean": float(np.mean(cv_results["test_precision"])),
    "cv_val_f1_mean": float(np.mean(cv_results["test_f1"])),
    "cv_fold_aucs": cv_results["test_roc_auc"].tolist(),
    "cv_fold_recalls": cv_results["test_recall"].tolist(),
}

print("5-fold stratified CV summary (training partition only):")
for k, v in cv_summary.items():
    print(f"  {k}: {v}")


In [ ]:
# ----------------------------------------------------------------------
# Full-training-set fit, evaluated ONCE on the untouched holdout test set.
# ----------------------------------------------------------------------
train_proba = logreg.predict_proba(X_train)[:, 1]
test_proba = logreg.predict_proba(X_test)[:, 1]
train_pred_05 = (train_proba >= 0.5).astype(int)
test_pred_05 = (test_proba >= 0.5).astype(int)

train_auc = roc_auc_score(y_train, train_proba)
test_auc = roc_auc_score(y_test, test_proba)

logreg_results_05 = {
    "train_auc": float(train_auc),
    "test_auc": float(test_auc),
    "train_recall_at_0.5": float(recall_score(y_train, train_pred_05)),
    "test_recall_at_0.5": float(recall_score(y_test, test_pred_05)),
    "train_precision_at_0.5": float(precision_score(y_train, train_pred_05, zero_division=0)),
    "test_precision_at_0.5": float(precision_score(y_test, test_pred_05, zero_division=0)),
    "test_brier": float(brier_score_loss(y_test, test_proba)),
}

print("Logistic regression, 0.5 decision threshold:")
for k, v in logreg_results_05.items():
    print(f"  {k}: {v}")

cm = confusion_matrix(y_test, test_pred_05)
print("\nHoldout confusion matrix [[TN FP] [FN TP]]:")
print(cm)


**Note on threshold tuning:** this model already uses
`class_weight="balanced"` at fit time, which shifts the decision boundary to
compensate for the imbalanced target. We tested additionally re-tuning the
classification threshold to the holdout set's own prevalence (6.82%) on top
of that weighting, and found it double-corrects for imbalance: it pushes
nearly every holdout row above the threshold, driving recall to 1.0 and
precision down to the base rate (a degenerate, all-positive classifier).
Principled threshold selection -- likely via a precision-recall operating
point chosen together with care teams -- is deferred to the hyperparameter
tuning phase next week, not attempted here.


## 5. Coefficient Table (Odds Ratios)

Coefficients are on the standardized/dummy-coded design matrix. A positive
coefficient means the feature increases the log-odds of 30-day readmission;
`exp(coefficient)` gives the odds ratio.

**Caution on rare categories:** some discharge-disposition and
admission-source categories have very few training encounters (single
digits, in a few cases). A large coefficient estimated from a handful of
rows is unstable and should not be over-interpreted as a strong clinical
signal -- the table below flags each feature's training-set category count
so this can be judged directly.


In [ ]:
coef_df = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": logreg.coef_[0],
})
coef_df["odds_ratio"] = np.exp(coef_df["coefficient"])
coef_df["abs_coef"] = coef_df["coefficient"].abs()

# Flag dummy features backed by very small category counts.
train_cat_counts = {}
for col in CATEGORICAL:
    for cat, n in train[col].value_counts().items():
        train_cat_counts[f"{col}_{cat}"] = n
coef_df["category_n"] = coef_df["feature"].map(train_cat_counts)

coef_df = coef_df.sort_values("abs_coef", ascending=False)
coef_df.to_csv("logreg_coefficients.csv", index=False)

print("Top 20 coefficients by magnitude (all categories):")
print(coef_df.head(20)[["feature", "coefficient", "odds_ratio", "category_n"]].to_string(index=False))

print("\n\nTop 15 STABLE coefficients (category_n >= 500, or continuous):")
stable = coef_df[(coef_df["category_n"].isna()) | (coef_df["category_n"] >= 500)]
print(stable.head(15)[["feature", "coefficient", "odds_ratio", "category_n"]].to_string(index=False))

print("\n\nCore continuous predictors (hypothesis-relevant):")
cont_feats = CONTINUOUS + ["age_ord"]
print(coef_df[coef_df["feature"].isin(cont_feats)][["feature", "coefficient", "odds_ratio"]].to_string(index=False))


## 6. Model Assumption Checks

Logistic regression carries three assumptions worth testing directly here:
(1) independence of observations, (2) no severe multicollinearity among
predictors, and (3) linearity of the logit for continuous predictors.


### 6.1 Independence of observations

Already addressed structurally in Section 1.12: the patient-level snapshot
retains exactly one row per patient, so no observation shares a patient with
another observation in the same partition. This was the specific
independence violation identified in the Preprocessing Report (23.4% of
patients had more than one encounter) and it is resolved by design here, not
tested statistically.


### 6.2 Multicollinearity (Variance Inflation Factor)

Rule of thumb: VIF > 5 is a concern, VIF > 10 indicates a real problem.


In [ ]:
X_vif = train[CONTINUOUS].copy()
X_vif = sm.add_constant(X_vif)
vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data[vif_data["feature"] != "const"]
vif_data.to_csv("vif_table.csv", index=False)

print("VIF table (continuous predictors):")
print(vif_data.to_string(index=False))


### 6.3 Linearity of the logit (Box-Tidwell test)

For each continuous predictor, this adds an `x * ln(x)` interaction term to
a logistic model. A statistically significant interaction term (p < 0.05)
indicates the raw, untransformed predictor violates the linear-logit
assumption -- i.e., its true relationship with log-odds of readmission is
not a straight line.


In [ ]:
bt_df = train[CONTINUOUS + [TARGET]].copy()
bt_results = []
for col in CONTINUOUS:
    # Box-Tidwell requires strictly positive values; shift zero-inflated
    # counts by 1 so log(x) is defined.
    x = bt_df[col].astype(float) + 1.0
    interaction = x * np.log(x)
    X_bt = sm.add_constant(pd.DataFrame({col: x, f"{col}_x_ln": interaction}))
    model = sm.Logit(bt_df[TARGET], X_bt)
    try:
        res = model.fit(disp=0)
        pval = res.pvalues[f"{col}_x_ln"]
        bt_results.append({
            "feature": col,
            "box_tidwell_pvalue": pval,
            "linear_logit_violated": bool(pval < 0.05),
        })
    except Exception as e:
        bt_results.append({
            "feature": col, "box_tidwell_pvalue": np.nan,
            "linear_logit_violated": f"fit_failed: {e}",
        })

bt_df_out = pd.DataFrame(bt_results)
bt_df_out.to_csv("box_tidwell_results.csv", index=False)

print("Box-Tidwell linearity-of-logit check:")
print(bt_df_out.to_string(index=False))


## 7. Save Summary Outputs

Everything needed for the write-up is collected into `model_summary.json`,
and holdout predictions are saved separately for the calibration plot below.


In [ ]:
summary = {
    "n_train": len(train),
    "n_test": len(test),
    "n_features": len(feature_columns),
    "train_prevalence": float(y_train.mean()),
    "test_prevalence": float(y_test.mean()),
    "dummy": dummy_results,
    "cv": cv_summary,
    "logreg_05": logreg_results_05,
    "confusion_matrix_test_05": cm.tolist(),
}
with open("model_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

fpr, tpr, thresh = roc_curve(y_test, test_proba)
pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": thresh}).to_csv("roc_curve.csv", index=False)
pd.DataFrame({"y_test": y_test, "test_proba": test_proba}).to_csv("test_predictions.csv", index=False)

print("Saved: model_summary.json, roc_curve.csv, test_predictions.csv, logreg_coefficients.csv, vif_table.csv, box_tidwell_results.csv")
print("\nFull summary:")
print(json.dumps(summary, indent=2))


## 8. Figures

In [ ]:
NAVY = "#1f3a5f"
TEAL = "#2a9d8f"
GRAY = "#888888"
RED = "#c0392b"

# --- Figure 1: ROC curve -------------------------------------------------
roc = pd.read_csv("roc_curve.csv")
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(roc["fpr"], roc["tpr"], color=NAVY, lw=2,
        label=f"Logistic regression (AUC = {logreg_results_05['test_auc']:.3f})")
ax.plot([0, 1], [0, 1], color=GRAY, lw=1.5, linestyle="--", label="Chance (AUC = 0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("Figure 1. ROC Curve\nBaseline Logistic Regression (Holdout Set)")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig("fig1_roc.png")
plt.show()


In [ ]:
# --- Figure 2: Coefficient plot (stable predictors only) -----------------
stable = coef_df[(coef_df["category_n"].isna()) | (coef_df["category_n"] >= 500)].copy()
stable = stable.sort_values("coefficient")
top_n = 15
plot_df = pd.concat([stable.head(top_n // 2), stable.tail(top_n - top_n // 2)]) if len(stable) > top_n else stable
plot_df = plot_df.sort_values("coefficient")

def clean_label(x):
    x = x.replace("discharge_disposition_id_label_", "Discharge: ")
    x = x.replace("admission_source_id_label_", "Admit source: ")
    x = x.replace("admission_type_id_label_", "Admit type: ")
    x = x.replace("diag_1_group_", "Primary dx: ")
    if len(x) > 38:
        x = x[:35] + "..."
    return x

plot_df["label"] = plot_df["feature"].apply(clean_label)

fig, ax = plt.subplots(figsize=(8, 6.5))
colors = [RED if v < 0 else TEAL for v in plot_df["coefficient"]]
ax.barh(plot_df["label"], plot_df["coefficient"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Coefficient (log-odds)")
ax.set_title("Figure 2. Top Stable Predictors by Coefficient Magnitude\n(categories with n >= 500 training encounters only)", fontsize=10.5)
fig.tight_layout()
fig.savefig("fig2_coefficients.png")
plt.show()


In [ ]:
# --- Figure 3: Calibration curve ------------------------------------------
preds = pd.read_csv("test_predictions.csv")
prob_true, prob_pred = calibration_curve(preds["y_test"], preds["test_proba"], n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(prob_pred, prob_true, marker="o", color=NAVY, lw=2, label="Logistic regression")
ax.plot([0, 1], [0, 1], color=GRAY, lw=1.5, linestyle="--", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability (within decile)")
ax.set_ylabel("Observed 30-day readmission rate")
ax.set_title("Figure 3. Calibration Curve\nHoldout Set (10 quantile bins)")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig("fig3_calibration.png")
plt.show()


In [ ]:
# --- Figure 4: CV stability / overfitting check ---------------------------
fold_aucs = cv_summary["cv_fold_aucs"]
fig, ax = plt.subplots(figsize=(6.5, 5))
x = np.arange(1, len(fold_aucs) + 1)
ax.plot(x, fold_aucs, marker="o", color=TEAL, lw=2, label="CV validation fold AUC")
ax.axhline(cv_summary["cv_train_auc_mean"], color=NAVY, lw=1.5, linestyle="--",
           label=f"Mean CV training-fold AUC ({cv_summary['cv_train_auc_mean']:.3f})")
ax.axhline(logreg_results_05["test_auc"], color=RED, lw=1.5, linestyle=":",
           label=f"Holdout test AUC ({logreg_results_05['test_auc']:.3f})")
ax.set_xlabel("Cross-validation fold")
ax.set_ylabel("AUC-ROC")
ax.set_xticks(x)
ax.set_ylim(0.55, 0.72)
ax.set_title("Figure 4. Cross-Validation Stability and\nTrain/Holdout Comparison")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig("fig4_cv_stability.png")
plt.show()


## Done

Copy the printed numbers from Sections 3-6 (dummy baseline, CV summary,
logistic regression metrics, confusion matrix, coefficient tables, VIF
table, and Box-Tidwell results) plus the four saved figures
(`fig1_roc.png`, `fig2_coefficients.png`, `fig3_calibration.png`,
`fig4_cv_stability.png`) back for the write-up.
